In [1]:
from db_conn import conn_db_wf
import polars as pl
engine, metadata = conn_db_wf()
conn = engine.connect()
query = """
        SELECT fecha, intervalo, tipo_medios, id_cola, nombre_cola, nombre_cola_dimension as nombre_cola_2, oferta, contestadas, abandonadas, saliente, flujo_saliente, velocidad_media_respuesta_seg, porcentaje_nivel_servicio, porcentaje_objetivo_nivel_servicio, espera_media_seg, manejo_medio_seg, conversacion_media_seg, retencion_media_seg, acw_medio_seg, abandono_medio_seg, manejo, retener, transferir, tiempo_manejo_total, cumplen_sla, tiempo_conversacion_total, tiempo_retencion_total, tiempo_acw_total, abandono_rapido, salientes_intentadas, nombre_campana, tmo_objetivo, nivel_atencion_objetivo, nivel_servicio_objetivo, tiempo_maximo_espera_objetivo, tipo_campana, cliente, unidad_negocio, jefe_negocio, supervisores
	FROM reporting.datos_colas
	WHERE plataforma_operacion = 'Genesys'
"""
df = pl.read_database(query, conn, infer_schema_length=None, batch_size=10000)

In [2]:
config_columnas = {
    "fechas": {
        "columnas": ["fecha"],
        "tipo": pl.Date,
        "formato": "dd-mm-yyyy",
    },
    "horas": {
        "columnas": ["intervalo"],
        "tipo": pl.Time,
        "formato": "hh:mm",
    },
    "textos": {
        "columnas": [
            "tipo_medios", "id_cola", "nombre_cola", "nombre_cola_2",
            "nombre_campana", "tipo_campana", "cliente", "unidad_negocio",
            "jefe_negocio", "supervisores",
        ],
        "tipo": pl.String,
        "formato": "@",
    },
    "enteros": {
        "columnas": [
            "oferta", "contestadas", "abandonadas", "saliente", "flujo_saliente",
            "manejo", "retener", "transferir", "cumplen_sla", "abandono_rapido",
            "salientes_intentadas", "tmo_objetivo", "tiempo_maximo_espera_objetivo",
        ],
        "tipo": pl.Int64,
        "formato": "#,##0",
    },
    "decimales": {
        "columnas": [
            "velocidad_media_respuesta_seg", "espera_media_seg", "manejo_medio_seg",
            "conversacion_media_seg", "retencion_media_seg", "acw_medio_seg",
            "abandono_medio_seg",
        ],
        "tipo": pl.Float64,
        "formato": "#,##0.00",
    },
    "porcentajes": {
        "columnas": [
            "porcentaje_nivel_servicio", "porcentaje_objetivo_nivel_servicio",
            "nivel_atencion_objetivo", "nivel_servicio_objetivo",
        ],
        "tipo": pl.Float64,
        "formato": "0.00%",
    },
    "duraciones": {
        "columnas": [
            "tiempo_manejo_total", "tiempo_conversacion_total",
            "tiempo_retencion_total", "tiempo_acw_total",
        ],
        "tipo": pl.Float64,
        "formato": "[h]:mm:ss",
    },
}


In [3]:
# Cada columna de la consulta debe aparecer una sola vez en el mapeo.
columnas_configuradas = [
    columna
    for grupo in config_columnas.values()
    for columna in grupo["columnas"]
]

if len(columnas_configuradas) != len(set(columnas_configuradas)):
    raise ValueError("Hay columnas repetidas en config_columnas.")

sin_configurar = sorted(set(df.columns) - set(columnas_configuradas))
fuera_de_query = sorted(set(columnas_configuradas) - set(df.columns))
if sin_configurar or fuera_de_query:
    raise ValueError(
        f"Columnas sin configurar: {sin_configurar}; "
        f"columnas fuera de la query: {fuera_de_query}"
    )

# Fijar los tipos incluso cuando toda una columna contiene nulos.
tipos_esperados = {
    columna: grupo["tipo"]
    for grupo in config_columnas.values()
    for columna in grupo["columnas"]
}
df = df.cast(tipos_esperados)

columnas_time = [
    nombre
    for nombre, dtype in df.schema.items()
    if dtype == pl.Time
]

# df conserva Time y las duraciones en horas decimales de la vista SQL.
# Excel recibe las horas como texto y las duraciones como fracciones de dia.
# No redondear estas fracciones: el formato controla lo que se muestra.
df_excel = df.with_columns(
    [pl.col(col).dt.strftime("%H:%M:%S").alias(col) for col in columnas_time]
    + [
        (pl.col(col) / 24).alias(col)
        for col in config_columnas["duraciones"]["columnas"]
    ]
)


In [4]:
import xlwings as xw

ruta = r".\archivos\reporte_colas.xlsx"
tamano_bloque = 10_000

with xw.App(visible=False, add_book=False) as app:
    # Crear (o, más adelante, abrir la plantilla) antes de cambiar el cálculo.
    libro = app.books.add()

    with app.properties(
        calculation="manual",
        screen_updating=False,
        display_alerts=False,
    ):
        hoja = libro.sheets[0]
        hoja.name = "Datos"

        # Encabezados
        hoja.range("A1").value = [df_excel.columns]

        # Adaptar los separadores del mapeo a los que usa esta instancia de Excel.
        separadores_excel = str.maketrans({
            ".": app.api.DecimalSeparator,
            ",": app.api.ThousandsSeparator,
        })

        # Formatos antes de escribir, para preservar los identificadores de texto.
        if df_excel.height > 0:
            for grupo in config_columnas.values():
                for columna in grupo["columnas"]:
                    indice_columna = df_excel.columns.index(columna) + 1
                    hoja.range(
                        (2, indice_columna),
                        (df_excel.height + 1, indice_columna),
                    ).number_format = grupo["formato"].translate(separadores_excel)

        # Datos por bloques
        for inicio in range(0, df_excel.height, tamano_bloque):
            bloque = df_excel.slice(inicio, tamano_bloque)

            datos = bloque.rows()

            fila_excel = inicio + 2

            hoja.range((fila_excel, 1)).value = datos

        # Rango completo
        rango_tabla = hoja.range(
            (1, 1),
            (df_excel.height + 1, df_excel.width)
        )

        # Crear tabla
        tabla = hoja.tables.add(
            source=rango_tabla,
            name="t_datos",
            has_headers=True,
            table_style_name="TableStyleMedium9"
        )

        tabla.show_table_style_first_column = False
        tabla.show_table_style_last_column = False
        tabla.show_table_style_row_stripes = True
        tabla.show_table_style_column_stripes = True

        libro.save(ruta)

    # Restaurar las propiedades antes de cerrar el último libro.
    libro.close()
